In [ ]:
! pip install accelerate bitsandbytes

In [ ]:
from transformers import AutoProcessor , PaliGemmaForConditionalGeneration , pipeline , BitsAndBytesConfig
import torch
from PIL import Image
import requests

In [ ]:
model_id = "google/paligemma-3b-mix-224"
device = "cuda:0"
dtype = torch.bfloat16

In [ ]:
## loading image
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/car.jpg?download=true"
image = Image.open(requests.get(url, stream=True).raw)

In [ ]:
## quantized config
quantization_config = BitsAndBytesConfig(load_in_4bit=True)
quantization_config

In [ ]:
model = PaliGemmaForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config = quantization_config
).eval()
processor = AutoProcessor.from_pretrained(model_id)

In [ ]:
prompt = "caption this image"
model_inputs = processor(text=prompt,images=image,return_tensors="pt").to(model.device)
input_len = model_inputs["input_ids"].shape[-1]

In [ ]:
with torch.inference_mode():
  generation = model.generate(**model_inputs,do_sample=False,max_new_tokens=100)
  generation = generation[0][input_len:]
  decode = processor.decode(generation,skip_special_tokens=True)
  print(f"[INFO] :\n{decode}")